# Can we predict whether a finding is malignant?

This notebook tests whether the six attributes in `data.csv` contain useful signal for predicting the binary `class` column (`1` = malignant, `0` = benign). Because malignancy is rare, accuracy alone is misleading; the primary metric is **average precision (PR AUC)**, with ROC AUC, recall, precision, F1, and balanced accuracy as supporting metrics.

A crucial data-quality issue is handled explicitly: many rows share the exact same feature vector. Cross-validation therefore keeps identical feature vectors in the same fold, preventing train/test leakage.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay,
    average_precision_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold, cross_validate, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
pd.set_option('display.max_columns', 20)
warnings.filterwarnings('ignore', category=UserWarning)

In [ ]:
DATA_PATH = Path('data.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Expected {DATA_PATH.resolve()}')

df = pd.read_csv(DATA_PATH)
TARGET = 'class'
if TARGET not in df.columns:
    raise ValueError(f'Missing target column: {TARGET}')
if not set(df[TARGET].dropna().unique()).issubset({0, 1}):
    raise ValueError('class must be binary and encoded as 0/1')

feature_cols = [c for c in df.columns if c != TARGET]
X = df[feature_cols]
y = df[TARGET].astype(int)
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
display(df.head())
display(df.dtypes.rename('dtype').to_frame().T)

## Data audit

We check missing values, class balance, duplicates, and whether identical feature vectors ever have conflicting labels.

In [ ]:
audit = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing': df.isna().sum(),
    'unique': df.nunique(dropna=False),
})
display(audit)

class_summary = y.value_counts().sort_index().rename('count').to_frame()
class_summary['percent'] = 100 * class_summary['count'] / len(y)
display(class_summary.style.format({'percent': '{:.2f}%'}))

ax = class_summary['count'].plot.bar(color=['#4C78A8', '#E45756'], rot=0, title='Class imbalance')
ax.set(xlabel='Class (1 = malignant)', ylabel='Rows')
for container in ax.containers:
    ax.bar_label(container, fmt='{:,.0f}')
plt.tight_layout()
plt.show()

In [ ]:
# Each exact feature vector defines a group. All members of a group stay in one fold.
feature_group = pd.util.hash_pandas_object(X, index=False).astype(str)
group_stats = (
    df.assign(_group=feature_group)
      .groupby('_group')[TARGET]
      .agg(rows='size', positives='sum', distinct_labels='nunique')
      .sort_values('rows', ascending=False)
)

duplicate_feature_rows = int(feature_group.duplicated().sum())
conflicting_groups = group_stats.query('distinct_labels > 1')
print(f'Exact duplicate rows (including class): {df.duplicated().sum():,}')
print(f'Unique feature vectors: {feature_group.nunique():,}')
print(f'Extra rows repeating a feature vector: {duplicate_feature_rows:,}')
print(f'Feature vectors with conflicting labels: {len(conflicting_groups):,}')
display(group_stats.head(10))
if len(conflicting_groups):
    print('Conflicting groups (same inputs, both labels):')
    display(conflicting_groups)

The repeated feature vectors matter. A standard random split could show a model an identical input during training and testing. Group-aware cross-validation gives a more honest estimate of performance on unseen feature patterns. A conflicting group also sets an irreducible limit: no model using only these columns can distinguish rows whose attributes are identical but whose labels differ.

In [ ]:
# Quick univariate view. Correlation is descriptive, not a measure of model quality.
corr = df.corr(numeric_only=True)[TARGET].drop(TARGET).sort_values()
display(corr.rename('Pearson correlation with class').to_frame())
ax = corr.plot.barh(color=['#4C78A8' if v < 0 else '#E45756' for v in corr])
ax.axvline(0, color='black', linewidth=0.8)
ax.set(title='Univariate association with malignancy', xlabel='Pearson correlation')
plt.tight_layout()
plt.show()

## Leakage-safe model comparison

We compare a prevalence-only dummy baseline, balanced logistic regression, and a balanced random forest. Five-fold `StratifiedGroupKFold` approximately preserves class balance while ensuring exact feature duplicates never cross folds. Preprocessing lives inside each pipeline so it is fitted only on training folds.

In [ ]:
models = {
    'Dummy baseline': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', DummyClassifier(strategy='prior')),
    ]),
    'Logistic regression': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(
            max_iter=2_000, class_weight='balanced', random_state=RANDOM_STATE
        )),
    ]),
    'Random forest': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', RandomForestClassifier(
            n_estimators=400, min_samples_leaf=2, class_weight='balanced',
            random_state=RANDOM_STATE, n_jobs=-1
        )),
    ]),
}

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    'PR AUC': 'average_precision',
    'ROC AUC': 'roc_auc',
    'Balanced accuracy': 'balanced_accuracy',
    'Precision': 'precision',
    'Recall': 'recall',
    'F1': 'f1',
}

rows = []
fold_results = {}
for model_name, model in models.items():
    scores = cross_validate(
        model, X, y, groups=feature_group, cv=cv, scoring=scoring, n_jobs=-1
    )
    fold_results[model_name] = scores
    for metric in scoring:
        values = scores[f'test_{metric}']
        rows.append({
            'model': model_name, 'metric': metric,
            'mean': values.mean(), 'std': values.std(ddof=1),
        })

cv_long = pd.DataFrame(rows)
cv_table = cv_long.pivot(index='model', columns='metric', values='mean')
cv_table = cv_table[list(scoring)]
display(cv_table.style.format('{:.3f}').highlight_max(axis=0, color='#D9EAD3'))

In [ ]:
# Mean ± one standard deviation across folds for the two threshold-free metrics.
plot_data = cv_long.query("metric in ['PR AUC', 'ROC AUC']")
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, metric in zip(axes, ['PR AUC', 'ROC AUC']):
    part = plot_data.query('metric == @metric').set_index('model').loc[list(models)]
    ax.barh(part.index, part['mean'], xerr=part['std'], color=['#B9B9B9', '#4C78A8', '#F58518'], capsize=4)
    ax.set(title=f'{metric} (grouped 5-fold CV)', xlabel='Score', xlim=(0, 1))
plt.tight_layout()
plt.show()

## Out-of-fold predictions

Every prediction below comes from a fold in which that row's feature group was absent from training. The default 0.5 threshold is shown for illustration; a clinical threshold must be selected from the relative harms of missed cancers and false alarms, then validated on independent data.

In [ ]:
oof_probabilities = {}
oof_rows = []
for model_name, model in models.items():
    prob = cross_val_predict(
        model, X, y, groups=feature_group, cv=cv, method='predict_proba', n_jobs=-1
    )[:, 1]
    pred = (prob >= 0.5).astype(int)
    oof_probabilities[model_name] = prob
    oof_rows.append({
        'model': model_name,
        'PR AUC': average_precision_score(y, prob),
        'ROC AUC': roc_auc_score(y, prob),
        'Balanced accuracy': balanced_accuracy_score(y, pred),
        'Precision @ 0.5': precision_score(y, pred, zero_division=0),
        'Recall @ 0.5': recall_score(y, pred, zero_division=0),
        'F1 @ 0.5': f1_score(y, pred, zero_division=0),
    })

oof_summary = pd.DataFrame(oof_rows).set_index('model')
display(oof_summary.style.format('{:.3f}'))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for model_name in ['Logistic regression', 'Random forest']:
    PrecisionRecallDisplay.from_predictions(y, oof_probabilities[model_name], name=model_name, ax=axes[0])
    RocCurveDisplay.from_predictions(y, oof_probabilities[model_name], name=model_name, ax=axes[1])
axes[0].axhline(y.mean(), color='gray', linestyle='--', label=f'Prevalence = {y.mean():.3f}')
axes[0].set_title('Out-of-fold precision–recall curves')
axes[0].legend()
axes[1].plot([0, 1], [0, 1], color='gray', linestyle='--')
axes[1].set_title('Out-of-fold ROC curves')
plt.tight_layout()
plt.show()

In [ ]:
best_name = cv_table['PR AUC'].idxmax()
best_prob = oof_probabilities[best_name]
best_pred = (best_prob >= 0.5).astype(int)
print(f'Best model by mean cross-validated PR AUC: {best_name}')
print(classification_report(y, best_pred, target_names=['benign', 'malignant'], digits=3, zero_division=0))
ConfusionMatrixDisplay(confusion_matrix(y, best_pred), display_labels=['benign', 'malignant']).plot(cmap='Blues')
plt.title(f'{best_name}: out-of-fold predictions at threshold 0.5')
plt.tight_layout()
plt.show()

## Which attributes contribute most?

The plots below are descriptive fits on the full dataset, not causal effects. Logistic coefficients show standardized linear associations; random-forest impurity importances show which variables the fitted forest used most.

In [ ]:
fitted_logistic = clone(models['Logistic regression']).fit(X, y)
fitted_forest = clone(models['Random forest']).fit(X, y)
importance = pd.DataFrame({
    'logistic coefficient': fitted_logistic.named_steps['model'].coef_[0],
    'forest importance': fitted_forest.named_steps['model'].feature_importances_,
}, index=feature_cols).sort_values('forest importance')
display(importance.sort_values('forest importance', ascending=False))
importance.plot.barh(subplots=True, figsize=(10, 4), legend=False, layout=(1, 2), title=['Logistic coefficient', 'Forest importance'])
plt.tight_layout()
plt.show()

## Conclusion

The cells below generate a data-driven conclusion from the leakage-safe cross-validation results. A model is useful for ranking risk if its PR AUC materially exceeds the malignancy prevalence (the random-ranking baseline), while ROC AUC measures overall ranking separation. This is evidence of predictability in this dataset—not evidence that the model is ready for clinical use.

In [ ]:
best_pr = cv_table.loc[best_name, 'PR AUC']
best_roc = cv_table.loc[best_name, 'ROC AUC']
prevalence = y.mean()
lift = best_pr / prevalence
print(
    f'YES — the attributes contain substantial predictive signal.\n'
    f'{best_name} achieved mean grouped-CV PR AUC {best_pr:.3f} '
    f'(vs. prevalence baseline {prevalence:.3f}, about {lift:.1f}× lift) '
    f'and ROC AUC {best_roc:.3f}.\n'
    'These estimates keep identical feature vectors in one fold, reducing duplicate leakage.'
)
print(
    '\nBefore deployment: validate on a truly independent, time/site-separated cohort; '
    'investigate the large repeated/conflicting feature group; identify the attributes and '
    'their collection process; select a threshold based on false-negative cost; assess '
    'calibration, subgroup performance, and confidence intervals. Do not use this notebook '
    'as a clinical diagnostic tool.'
)